# Deep learning II: the shallow network in PyTorch

COSC 3430 Artificial Intelligence | Fall 26 | Augustana U.

The network of the lecture with one input, three hidden units and one output, built in PyTorch: the same loss as with our own function, then training of all ten parameters, a wider network, and different starting values. There is nothing to type - run the cells in order and compare the output with the lecture.

Later cells use the variables and functions of earlier ones: Runtime > Run all runs everything at once. The notebook needs nothing from the first notebook of this lecture.

## Code-along 1. Setup and training data (demo)

PyTorch is a library for neural networks. Its array type is the *tensor*: like a NumPy array, but PyTorch can also compute how the loss changes when a parameter changes. `reshape(-1, 1)` makes a column: one row per training point, one column per input.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# PyTorch and its module with the building blocks of networks
import torch
from torch import nn

# the twenty training points of the lecture
x_train = np.array([0.09291784,0.46809093,0.93089486,0.67612654,0.73441752,
                    0.86847339,0.49873225,0.51083168,0.18343972,0.99380898,
                    0.27840809,0.38028817,0.12055708,0.56715537,0.92005746,
                    0.77072270,0.85278176,0.05315950,0.87168699,0.58858043])
y_train = np.array([-0.15934537,0.18195445,0.45127015,0.13921448,0.09366691,
                    0.30567674,0.37229117,0.40716968,-0.08131792,0.41187806,
                    0.36943738,0.3994327,0.01906257,0.35820410,0.45256496,
                    -0.0183121,0.02957665,-0.24354444,0.14803884,0.26824970])

# as tensors: one row per training point, one column
X = torch.tensor(x_train, dtype=torch.float32).reshape(-1, 1)
Y = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
# 20 rows, 1 column each
print(X.shape, Y.shape)

## Code-along 2. Build the network (demo)

`nn.Sequential` chains three steps: `nn.Linear(1, 3)` computes the three preactivations, `nn.ReLU()` the activations, `nn.Linear(3, 1)` the weighted sum with its offset.

In [ ]:
# three steps: preactivations, activations, weighted sum
model = nn.Sequential(
    nn.Linear(1, 3),
    nn.ReLU(),
    nn.Linear(3, 1),
)
# the network, step by step
print(model)

## Code-along 3. The parameters (demo)

PyTorch calls a slope or weight a `weight` and an offset a `bias`. The number in a name is the step in the chain.

| PyTorch | Lecture |
|---|---|
| `0.weight` | θ₁₁, θ₂₁, θ₃₁ |
| `0.bias` | θ₁₀, θ₂₀, θ₃₀ |
| `2.weight` | φ₁, φ₂, φ₃ |
| `2.bias` | φ₀ |

In [ ]:
# every group of parameters: name, shape, number of values
for name, p in model.named_parameters():
    print(name, tuple(p.shape), p.numel())
# all parameters of the network
print('Parameters:', sum(p.numel() for p in model.parameters()))

## Code-along 4. Load the parameters of the lecture (demo)

A new network starts with random parameters. Replace them with the ten values of the fit by hand. `model[0]` is the first step, `model[2]` the last.

In [ ]:
with torch.no_grad():
    # input side: slopes theta_d1 and offsets theta_d0
    model[0].weight[:] = torch.tensor([[-1.0], [2.0], [0.65]])
    model[0].bias[:] = torch.tensor([0.3, -1.0, -0.5])
    # output side: weights phi_1..phi_3 and offset phi_0
    model[2].weight[:] = torch.tensor([[-2.5, -0.7, 5.3]])
    model[2].bias[:] = torch.tensor([0.4])

## Code-along 5. The same loss (demo)

`nn.MSELoss(reduction='sum')` is the least-squares loss of the lecture: the sum of the squared differences. The loss must be 0.168, as with `shallow_1_1_3`.

In [ ]:
# least-squares loss: sum of the squared differences
loss_fn = nn.MSELoss(reduction='sum')
# predictions for the 20 training inputs
y_predict = model(X)
loss = loss_fn(y_predict, Y)
print(f'Loss = {loss.item():.3f}')

## Code-along 6. A training step (demo)

One training step: run the network and compute the loss, compute for every parameter in which direction the loss falls (`loss.backward()`), and change all parameters by a small step (`optimizer.step()`). `lr` is the learning rate, the size of a step.

In [ ]:
def train(model, X, Y, steps=2000, lr=0.01):
    """Train the model on (X, Y); return the final loss"""
    # the optimizer changes the parameters; lr is the step size
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps + 1):
        # run the network and compute the loss
        loss = loss_fn(model(X), Y)
        if step % 500 == 0:
            print(f'Step {step:5d}: loss = {loss.item():.3f}')
        # clear the old directions, compute the new ones
        optimizer.zero_grad()
        loss.backward()
        # change every parameter by a small step
        optimizer.step()
    return loss_fn(model(X), Y).item()

## Code-along 7. Train from the fit by hand (demo)

Train for 2000 steps, starting from the fit by hand.

In [ ]:
# 2000 training steps, starting from the fit by hand
loss = train(model, X, Y)

Read the parameters back. The joint of a hidden unit is where its preactivation is zero: x* = -θ_d0 / θ_d1. By hand the joints were at 0.3, 0.5 and 0.77.

In [ ]:
# the trained parameters as NumPy arrays
theta_1 = model[0].weight.detach().numpy().flatten()   # slopes
theta_0 = model[0].bias.detach().numpy()               # offsets
phi = model[2].weight.detach().numpy().flatten()       # weights
phi_0 = model[2].bias.detach().numpy()                 # offset
print('theta_d0:', theta_0.round(2))
print('theta_d1:', theta_1.round(2))
print('phi_0:   ', phi_0.round(2), ' phi_1..3:', phi.round(2))
# where each hidden unit switches
print('Joints:  ', (-theta_0 / theta_1).round(2))

## Code-along 8. The trained network (demo)

`plot_model` runs the model on 100 inputs between 0 and 1 and draws the output with the training points on top.

In [ ]:
def plot_model(model, title):
    """Network output over [0, 1] with the training points (magenta)"""
    # 100 inputs between 0 and 1, as a column
    x = torch.linspace(0, 1, 100).reshape(-1, 1)
    # inference only
    with torch.no_grad():
        y = model(x)
    fig, ax = plt.subplots()
    ax.plot(x.numpy(), y.numpy())
    ax.plot(x_train, y_train, 'mo')
    ax.set_xlabel('Input, $x$'); ax.set_ylabel('Output, $y$')
    ax.set_xlim([0, 1]); ax.set_ylim([-1, 1])
    ax.set_title(title)

In [ ]:
# the trained network with three hidden units
plot_model(model, f'D = 3, trained: loss = {loss:.3f}')
plt.show()

## Code-along 9. More hidden units (demo)

How many parameters does a network with D = 50 hidden units have? Count before you run the cell. This network trains from a random start; the seed gives everyone the same start.

In [ ]:
# same random starting parameters for everyone
torch.manual_seed(0)
# number of hidden units
D = 50
wide = nn.Sequential(nn.Linear(1, D), nn.ReLU(), nn.Linear(D, 1))
print('Parameters:', sum(p.numel() for p in wide.parameters()))
# 5000 training steps from the random start
loss_wide = train(wide, X, Y, steps=5000)

In [ ]:
# the trained network with 50 hidden units
plot_model(wide, f'D = 50, trained: loss = {loss_wide:.3f}')
plt.show()

## Code-along 10. The starting values matter (demo)

Three hidden units again, from four different random starts. Compare the four losses with the 0.142 from the fit by hand and with 0.645, the loss of the best line.

In [ ]:
import contextlib, io
for seed in range(4):
    # a new network with its own random start
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(1, 3), nn.ReLU(), nn.Linear(3, 1))
    # train without printing the steps
    with contextlib.redirect_stdout(io.StringIO()):
        final = train(net, X, Y, steps=5000)
    print(f'Seed {seed}: loss = {final:.3f}')